In [1]:
from datetime import datetime
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import time

try:
    client = MongoClient('mongodb://bigdata_mongodb:27017/')
    db = client['proyecto_bigdata']
    coleccion = db['alojamientos']
    print("Conexion a MongoDB exitosa!")
except Exception as e:
    print(f"Error conectando a MongoDB: {e}")

Conexion a MongoDB exitosa!


In [2]:
# Ciudades de Norte a Sur de Chile (sin sesgo de zona)
ciudades = [
    'Arica', 'Iquique', 'Calama', 'Antofagasta',
    'Copiapo', 'La Serena',
    'Valparaiso', 'Vina del Mar', 'Santiago', 'Rancagua',
    'Talca', 'Chillan', 'Concepcion', 'Temuco',
    'Valdivia', 'Puerto Varas', 'Puerto Montt',
    'Coyhaique', 'Puerto Natales', 'Punta Arenas'
]

def limpiar_precio(texto):
    numeros = ''
    for c in texto:
        if c.isdigit():
            numeros += c
    if not numeros:
        return None
    precio = float(numeros)
    if precio < 5000 or precio > 10000000:
        return None
    return precio

def obtener_estrellas(hotel):
    try:
        stars = hotel.find_elements(
            By.CSS_SELECTOR,
            '[data-testid="rating-stars"] span.fc70cba028.bdc459fcb4.f24706dc71:not(.e2cec97860)'
        )
        if stars:
            return len(stars)
        star_div = hotel.find_element(By.CSS_SELECTOR, '[data-testid="rating-stars"]')
        parent = star_div.find_element(By.XPATH, '..')
        label = parent.get_attribute('aria-label')
        if label and 'de 5' in label:
            return int(label.split(' de 5')[0].strip())
    except:
        pass
    return 0

def obtener_tipo(hotel):
    try:
        desc = hotel.find_element(By.CSS_SELECTOR, '.fff1944c52').text.lower()
        nombre = hotel.find_element(By.CSS_SELECTOR, '[data-testid="title"]').text.lower()
        texto = desc + ' ' + nombre
        if 'apart' in texto or 'apartamento' in texto:
            return 'apartamento'
        elif 'hostal' in texto or 'hostel' in texto:
            return 'hostal'
        elif 'cabaña' in texto or 'cabana' in texto:
            return 'cabana'
        elif 'lodge' in texto:
            return 'lodge'
        elif 'camping' in texto:
            return 'camping'
        elif 'domo' in texto:
            return 'domo'
        elif 'hotel' in texto:
            return 'hotel'
        else:
            return 'otro'
    except:
        return 'otro'

def determinar_zona(ciudad):
    if ciudad in ['Arica', 'Iquique', 'Calama', 'Antofagasta']:
        return 'Norte Grande'
    elif ciudad in ['Copiapo', 'La Serena']:
        return 'Norte Chico'
    elif ciudad in ['Valparaiso', 'Vina del Mar', 'Santiago', 'Rancagua']:
        return 'Centro'
    elif ciudad in ['Talca', 'Chillan', 'Concepcion', 'Temuco']:
        return 'Centro Sur'
    elif ciudad in ['Valdivia', 'Puerto Varas', 'Puerto Montt']:
        return 'Los Lagos'
    else:
        return 'Patagonia'

def configurar_driver():
    opciones = Options()
    opciones.add_argument('--no-sandbox')
    opciones.add_argument('--disable-dev-shm-usage')
    opciones.add_argument('--disable-blink-features=AutomationControlled')
    opciones.add_experimental_option('excludeSwitches', ['enable-automation'])
    opciones.add_experimental_option('useAutomationExtension', False)
    opciones.add_argument(
        'user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    )
    opciones.binary_location = '/usr/bin/google-chrome'
    driver = webdriver.Chrome(
        service=Service('/usr/bin/chromedriver'),
        options=opciones
    )
    driver.execute_script(
        "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    )
    return driver

def scraper_booking(ciudad):
    url = (
        f'https://www.booking.com/searchresults.es-ar.html?'
        f'ss={ciudad.replace(" ", "+")}%2C+Chile'
        f'&order=popularity'
    )

    print(f'\n{"="*50}')
    print(f'Ciudad: {ciudad}')
    print(f'{"="*50}')

    driver = None
    try:
        driver = configurar_driver()
        driver.get(url)
        time.sleep(6)

        print('\n>>> ACCION REQUERIDA <<<')
        print('1. Abre: localhost:6080/vnc.html')
        print('2. Verifica que cargaron alojamientos con precios')
        print('3. Si hay captcha, resuelvelo manualmente')
        print('4. Cuando todo se vea bien, vuelve aqui')
        input('>>> Presiona ENTER para comenzar a extraer datos <<<\n')

        time.sleep(2)

        hoteles = driver.find_elements(
            By.CSS_SELECTOR, '[data-testid="property-card"]'
        )

        if not hoteles:
            print(f'Sin resultados para {ciudad}')
            return 0

        print(f'Alojamientos encontrados: {len(hoteles)}')
        guardados = 0
        sin_precio = 0

        for i, hotel in enumerate(hoteles):
            try:
                driver.execute_script(
                    "arguments[0].scrollIntoView({block: 'center'});", hotel
                )
                time.sleep(0.8)

                try:
                    nombre = hotel.find_element(
                        By.CSS_SELECTOR, '[data-testid="title"]'
                    ).text.strip()
                except:
                    continue

                if not nombre:
                    continue

                precio = None
                selectores_precio = [
                    '[data-testid="price-and-discounted-price"]',
                    '[data-testid="price"]',
                    '.prco-valign__middle-helper',
                    '[data-testid="availability-rate-information"]',
                ]
                for selector in selectores_precio:
                    try:
                        elem = hotel.find_element(By.CSS_SELECTOR, selector)
                        texto = elem.text.strip()
                        if texto:
                            precio = limpiar_precio(texto)
                            if precio:
                                break
                    except:
                        continue

                if not precio:
                    sin_precio += 1
                    print(f'  [{i+1}] SIN PRECIO: {nombre[:40]}')
                    precio = 0.0
                else:
                    print(f'  [{i+1}] ${precio:,.0f} | {nombre[:40]}')

                puntuacion = None
                try:
                    punt_elem = hotel.find_element(
                        By.CSS_SELECTOR, '[data-testid="review-score"]'
                    )
                    punt_texto = punt_elem.text.strip()
                    for palabra in punt_texto.replace('\n', ' ').split():
                        try:
                            val = float(palabra.replace(',', '.'))
                            if 1.0 <= val <= 10.0:
                                puntuacion = val
                                break
                        except:
                            continue
                except:
                    puntuacion = None

                estrellas = obtener_estrellas(hotel)
                tipo = obtener_tipo(hotel)
                zona = determinar_zona(ciudad)

                try:
                    url_hotel = hotel.find_element(
                        By.CSS_SELECTOR, '[data-testid="title-link"]'
                    ).get_attribute('href')
                    url_hotel = url_hotel.split('?')[0] if '?' in url_hotel else url_hotel
                except:
                    url_hotel = url

                registro = {
                    'nombre_hotel': nombre,
                    'precio_noche': precio,
                    'ciudad': ciudad,
                    'zona_geografica': zona,
                    'estrellas': estrellas,
                    'tipo_alojamiento': tipo,
                    'puntuacion': puntuacion,
                    'fecha_captura': datetime.now(),
                    'url_origen': url_hotel,
                    'plataforma': 'Booking.com',
                    'integrante': 'camila-rojas',
                    'grupo': 'G5_Turismo_Hoteleria'
                }

                coleccion.update_one(
                    {
                        'nombre_hotel': nombre,
                        'ciudad': ciudad,
                        'plataforma': 'Booking.com'
                    },
                    {'$set': registro},
                    upsert=True
                )
                guardados += 1

            except Exception as e:
                print(f'  Error alojamiento {i+1}: {str(e)[:50]}')
                continue

        print(f'\nResumen {ciudad}:')
        print(f'  Guardados:  {guardados}')
        print(f'  Sin precio: {sin_precio}')
        return guardados

    except Exception as e:
        print(f'Error general en {ciudad}: {e}')
        return 0
    finally:
        if driver:
            driver.quit()


# Ejecutar
total_antes = coleccion.count_documents({'plataforma': 'Booking.com'})
print(f'Registros en MongoDB antes: {total_antes}')
print(f'Ciudades a procesar: {len(ciudades)}')

total_nuevos = 0
for ciudad in ciudades:
    nuevos = scraper_booking(ciudad)
    total_nuevos += nuevos
    if ciudad != ciudades[-1]:
        print(f'\nEsperando 15 segundos antes de la siguiente ciudad...')
        time.sleep(15)

total_despues = coleccion.count_documents({'plataforma': 'Booking.com'})
print(f'\n{"="*50}')
print(f'SCRAPING COMPLETADO')
print(f'{"="*50}')
print(f'Registros antes:         {total_antes}')
print(f'Registros ahora:         {total_despues}')
print(f'Nuevos/actualizados:     {total_despues - total_antes}')
print(f'{"="*50}')

Registros en MongoDB antes: 195
Ciudades a procesar: 20

Ciudad: Arica
Error general en Arica: Message: Unable to obtain driver for chrome; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors/driver_location


Esperando 15 segundos antes de la siguiente ciudad...

Ciudad: Iquique
Error general en Iquique: Message: Unable to obtain driver for chrome; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors/driver_location


Esperando 15 segundos antes de la siguiente ciudad...

Ciudad: Calama
Error general en Calama: Message: Unable to obtain driver for chrome; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors/driver_location


Esperando 15 segundos antes de la siguiente ciudad...

Ciudad: Antofagasta
Error general en Antofagasta: Message: Unable to obtain driver for chrome; For documentation

In [ ]:
print(coleccion.count_documents({'integrante': 'Camila-rojas'}))

In [4]:
import os

rutas_chrome = [
    '/usr/bin/google-chrome',
    '/usr/bin/chromium',
    '/usr/bin/chromium-browser',
    '/usr/lib/chromium/chromium',
    '/opt/google/chrome/chrome',
]

rutas_driver = [
    '/usr/bin/chromedriver',
    '/usr/lib/chromium/chromedriver',
    '/usr/local/bin/chromedriver',
    '/opt/google/chrome/chromedriver',
]

print("=== CHROME ===")
for r in rutas_chrome:
    print(f"{'✅' if os.path.exists(r) else '❌'} {r}")

print("\n=== CHROMEDRIVER ===")
for r in rutas_driver:
    print(f"{'✅' if os.path.exists(r) else '❌'} {r}")

=== CHROME ===
❌ /usr/bin/google-chrome
❌ /usr/bin/chromium
❌ /usr/bin/chromium-browser
❌ /usr/lib/chromium/chromium
❌ /opt/google/chrome/chrome

=== CHROMEDRIVER ===
❌ /usr/bin/chromedriver
❌ /usr/lib/chromium/chromedriver
❌ /usr/local/bin/chromedriver
❌ /opt/google/chrome/chromedriver


In [5]:
import subprocess
result = subprocess.run(
    ['apt-get', 'install', '-y', 'chromium', 'chromium-driver'],
    capture_output=True, text=True
)
print(result.stdout[-2000:])
print(result.stderr[-500:])


E: Could not open lock file /var/lib/dpkg/lock-frontend - open (13: Permission denied)
E: Unable to acquire the dpkg frontend lock (/var/lib/dpkg/lock-frontend), are you root?



In [6]:
import subprocess
result = subprocess.run(
    ['sudo', 'apt-get', 'install', '-y', 'chromium', 'chromium-driver'],
    capture_output=True, text=True
)
print(result.stdout[-2000:])
print(result.stderr[-500:])



sudo: a terminal is required to read the password; either use the -S option to read from standard input or configure an askpass helper
sudo: a password is required

